# Constructing the study cohort and funding categories

This notebook documents how the analytical cohort was built from PubMed/PKG records and institutionally licensed Web of Science (WoS) records. It covers disease screening, U.S. affiliation screening, publication matching, funding acknowledgments, and assignment of funding structure. The WoS inputs are restricted and are not included in this repository.


## Identify disease-related PubMed records

Build MeSH lookup information, identify records with disease terms in category C or F03, and screen affiliations for U.S. participation. The code reads the local PubMed source files and produces intermediate PMID-level records.


In [ ]:
disease_dict = {}
i=0
j=0
# i = 5032
# j = 15572
for index, row in mesh_data.iterrows():
    if pd.isna(row['MN']):
        continue
    label = ''
    if 'D' in row['MN']:
        label = 'D'
    if 'C' in row['MN']:
        label = 'C'
    if label=='':
        continue
    i=i+1
    #print(row['UI'], row['MN'])
    if row['UI'] not in disease_dict:
        if label=='C':
            j = j+1
        disease_dict[row['UI']] = label
    else:
        print(row['UI'])

In [ ]:
#pubmed_dict = {}
m = 0
n = 0
for file in pubmed_list:
    n=n+1
    if n<1:
        continue
    start = time.time()
    with gzip.open(pubmed_path+'/'+file, mode="rt",encoding='utf-8') as f:
        data = f.read()

    root = ET.fromstring(data)

    article_lists = root.findall('PubmedArticle')
    m = m+len(article_lists)
    for article in article_lists:

        ### Get PMID
        for x in article.iter('PMID'):
            pmid = x.text
            break

        ### check whether in USA
        is_usa = 'No'
        for x in article.iter('Affiliation'):
            aff = x.text
            if pd.isna(aff):
                continue
            if 'USA' in aff:
                is_usa = 'Yes'
                break
            if 'United States' in aff:
                is_usa = 'Yes'
                break
            if 'UNITED STATES' in aff:
                is_usa = 'Yes'
                break
        
        ### Check whether disease
        label = 'No_CD'
        is_mesh = 0
        for x in article.iter('MeshHeading'):
            is_mesh = 1
            ui = x.find('DescriptorName').attrib['UI']
            if ui in disease_dict:
                if disease_dict[ui]=='C':
                    label = 'C'
                    break
                if disease_dict[ui]=='D':
                    label = 'D'
        if is_mesh==0:
            label = 'None'
        pubmed_dict[pmid] = [is_usa,label]
    end = time.time()
    print(file, end-start)


In [ ]:
with open('z_states.csv') as f:
    line_0 = f.readline()

    while True:
        line = f.readline()
        if line=='':
            break
        line_list = line[:-1].split(',')
        state_list.append(line_list[0][1:-1])
        state_list.append(line_list[1][1:-1])

In [ ]:
m = 0
n = 0
for file in pubmed_list:
    n=n+1
    if n<0:
        continue
    start = time.time()
    with gzip.open(pubmed_path+'/'+file, mode="rt",encoding='utf-8') as f:
        data = f.read()

    root = ET.fromstring(data)

    article_lists = root.findall('PubmedArticle')
    m = m+len(article_lists)
    for article in article_lists:

        ### Get PMID
        for x in article.iter('PMID'):
            pmid = x.text
            break

        ### check whether in USA
        is_usa = 'No'
        for x in article.iter('Affiliation'):
            aff = x.text
            if pd.isna(aff):
                continue
            for y in us_list:
                if y=='US':
                    if ' '+y+',' in aff:
                        is_usa = 'Country'
                        break
                    if ' '+y+'.' in aff:
                        is_usa = 'Country'
                        break
                    if ' '+y+';' in aff:
                        is_usa = 'Country'
                        break
                else:
                    if y in aff:
                        is_usa = 'Country'
                        break
            if is_usa == 'Country':
                break
            for y in state_list:
                if ', '+y+',' in aff:
                    is_usa = 'State'
                    break
                if ', '+y+'.' in aff:
                    is_usa = 'State'
                    break
                if ', '+y+';' in aff:
                    is_usa = 'State'
                    break
                if ', '+y+' ' in aff:
                    is_usa = 'State'
            if is_usa!='No':
                break
        
        data_pubmed_dict[pmid]['is_us_2'] = is_usa
    end = time.time()
    print(file, end-start)

In [ ]:
from collections import defaultdict

def etree_to_dict(t):
    d = {t.tag: {} if t.attrib else None}
    children = list(t)
    if children:
        dd = defaultdict(list)
        for dc in map(etree_to_dict, children):
            for k, v in dc.items():
                dd[k].append(v)
        d = {t.tag: {k:v[0] if len(v) == 1 else v for k, v in dd.items()}}
    if t.attrib:
        d[t.tag].update(('@' + k, v) for k, v in t.attrib.items())
    if t.text:
        text = t.text.strip()
        if children or t.attrib:
            if text:
              d[t.tag]['#text'] = text
        else:
            d[t.tag] = text
    return d

In [ ]:
m = 0
n = 0
with open('2_pubmed.json','w',encoding='utf-8') as f0:
    for file in pubmed_list:
        n=n+1
        if n<0:
            continue
        start = time.time()
        with gzip.open(pubmed_path+'/'+file, mode="rt",encoding='utf-8') as f:
            data = f.read()

        root = ET.fromstring(data)

        article_lists = root.findall('PubmedArticle')
        m = m+len(article_lists)
        for article in article_lists:

            ### Get PMID
            for x in article.iter('PMID'):
                pmid = x.text
                break

            ### Check whether have grant and mesh data
            have_grant = 'No'
            for x in article.iter('Grant'):
                have_grant = 'Yes'
                break
            data_pubmed_dict[pmid]['have_grant'] = have_grant


            have_mesh = 'No'
            for x in article.iter('MeshHeadingList'):
                have_mesh = 'Yes'
                break
            data_pubmed_dict[pmid]['have_mesh'] = have_mesh


            if pmid in data_pubmed_dict:
                if data_pubmed_dict[pmid]['is_us_2']=='No':
                    continue
                article_dict = etree_to_dict(article)
                article_json = {"PMID":pmid}
                if 'GrantList' in article_dict['PubmedArticle']['MedlineCitation']['Article'].keys():
                    grant_data = article_dict['PubmedArticle']['MedlineCitation']['Article']['GrantList']['Grant']
                    article_json['Grant'] = [grant_data] if isinstance(grant_data, dict) else grant_data
                else:
                    article_json['Grant'] = None
                if 'MeshHeadingList' in article_dict['PubmedArticle']['MedlineCitation'].keys():
                    mesh_data = article_dict['PubmedArticle']['MedlineCitation']['MeshHeadingList']['MeshHeading']
                    article_json['MeshHeadingList'] = [mesh_data] if isinstance(mesh_data, dict) else mesh_data
                else:
                    article_json['MeshHeadingList'] = None
                f0.write(json.dumps(article_json) + "\n")

        end = time.time()
        print(file, end-start)

## Match publications and extract funding information

Match PubMed identifiers to WoS records, then extract funding organization names from the matched records. WoS source files and extracted publication-level records are not distributed here.


In [ ]:
for file in wos_list:
    i = 0
    year = file[3:7]
    if int(year)<i:
        continue

    start = time.time()
    with gzip.open(wos_path+'/'+file, mode="rt") as f:
        data = [json.loads(line) for line in f]
 
    for paper in data:
        pid = paper['id']
        pmid = ''
        have_grants = 'No'
        if paper['grants']!=None:
             have_grants = 'Yes'
        if paper['identifiers']!=None:
                for identifier in paper['identifiers']:
                    if identifier['type']=='pmid':
                        pmid = identifier['value']
                        if pmid in pubmed_dict:
                             pubmed_dict[pmid]['in_wos']='Yes'
                             pubmed_dict[pmid]['have_grants']=have_grants
                             pubmed_dict[pmid]['file'] = file
    end = time.time()
    print(file, end-start)


In [ ]:
with open('2_wos.json','w',encoding='utf-8') as f0:
    for file in wos_list:
        i = 0
        year = file[3:7]
        if int(year)<i:
            continue

        start = time.time()
        with gzip.open(wos_path+'/'+file, mode="rt") as f:
            data = [json.loads(line) for line in f]
    
        for paper in data:
            pid = paper['id']
            pmid = ''
            have_ftext = 'No'
            if paper['fund_text']!=None:
                have_ftext = 'Yes'
            if paper['identifiers']!=None:
                    for identifier in paper['identifiers']:
                        if identifier['type']=='pmid':
                            pmid = identifier['value']

                            if pmid in pubmed_dict:
                                pubmed_dict[pmid]['have_ftext']=have_ftext
                                paper_json = {'PMID':pmid, 'fund_text':paper['fund_text'],"grants":paper['grants']}
                                f0.write(json.dumps(paper_json) + "\n")


        end = time.time()
        print(file, end-start)

In [ ]:
input_path = "2_wos.json"
output_path = "2_wos_uscg.json"

with open(input_path, "r", encoding="utf-8") as fin, open(output_path, "w", encoding="utf-8") as fout:
    for i, line in enumerate(fin):
        item = json.loads(line)
        if item['PMID'] in id_set:
            fout.write(json.dumps(item) + "\n")
        if i % 10000 == 0:
            print(f"Processed {i} lines")

In [ ]:
from collections import defaultdict
id_to_lines = defaultdict(list)


file_path = "2_wos_uscg.json"
output_path = "2_wos_uscg_duplicates.json"

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        obj_id = obj["PMID"]
        id_to_lines[obj_id].append(obj)


with open(output_path, "w", encoding="utf-8") as out:
    for obj_id, records in id_to_lines.items():
        if len(records) > 1:
            for record in records:
                out.write(json.dumps(record) + "\n")

print(f"Total duplicate IDs: {sum(1 for v in id_to_lines.values() if len(v) > 1)}")

In [ ]:

with open("2_wos_uscg.json", "r") as f:
    data = [json.loads(line) for line in f]


with open("2_wos_uscg_duplicates.json", "r") as f:
    dup_pmids = {json.loads(line)['PMID'] for line in f}


pmid_to_index = {}


for idx, entry in enumerate(data):
    pmid = entry.get("PMID")
    if pmid in dup_pmids:
        pmid_to_index[pmid] = idx


unique_data = []
for idx, entry in enumerate(data):
    pmid = entry.get("PMID")
    if pmid not in dup_pmids:
        unique_data.append(entry)
    elif pmid_to_index[pmid] == idx:
        unique_data.append(entry)


with open("2_wos_uscg_deduplicated.json", "w") as f:
    for entry in unique_data:
        f.write(json.dumps(entry) + "\n")

## Include mental disorders and apply cohort filters

Add the F03 mental-disorder category and retain eligible U.S. disease-related papers with funding information. This stage also removes duplicate matched records.


In [ ]:
disease_dict = {}
i=0
j=0
# i = 5032
# j = 15572
for index, row in mesh_data.iterrows():
    m = 0
    n = 0
    if pd.isna(row['MN']):
        continue
    label = ''
    if 'F03' in row['MN']:
        label = 'F03'
        m=1
    if 'C' in row['MN']:
        label = 'C'
        n=1
    if m+n==0:
        continue
    i=i+1
    #print(row['UI'], row['MN'])
    if row['UI'] not in disease_dict:
        if label=='C':
            j = j+1
        disease_dict[row['UI']] = [label,m,n]
    else:
        print(row['UI'])

In [ ]:

input_file = '2_pubmed_v2.json'
results = []
i=0
with open(input_file, 'r', encoding='utf-8') as fin:
    for line in fin:
        obj = json.loads(line)
        pmid = obj.get("PMID")
        mesh_list = obj.get("MeshHeadingList", [])
        i=i+1
        if i==3:
            1#break
        mesh_ids = []
        if mesh_list is None:
            continue
        for mh in mesh_list:
            descriptor = mh.get("DescriptorName", {})
            ui = descriptor.get("@UI")
            if ui:
                mesh_ids.append(ui)

        have_c = any(disease_dict.get(mid, [None])[0] == "C" for mid in mesh_ids if mid in disease_dict)
        have_f03 = any(disease_dict.get(mid, [None])[0] == "F03" for mid in mesh_ids if mid in disease_dict)

        if have_c or have_f03:
            results.append({
                "PMID": pmid,
                "have_c": "yes" if have_c else "no",
                "have_f03": "yes" if have_f03 else "no"
            })


df_cf = pd.DataFrame(results)


# df.to_csv("labeled_pmids.csv", index=False)
# print(df.head())

In [ ]:
pmid_set = set(merged_df_cfg["PMID"].astype(str))

input_file = "2_pubmed_v2.json"
output_file = "41_pubmed_uscfg.json"

with open(input_file, "r", encoding="utf-8") as fin, open(output_file, "w", encoding="utf-8") as fout:
    i=0
    for line in fin:
        obj = json.loads(line)
        pmid = str(obj.get("PMID"))
        if pmid in pmid_set:
            i=i+1
            if i % 10000 == 0:
                print(f"Processed {i} lines")
            fout.write(line)

In [ ]:
pmid_set = set(merged_df_cfg["PMID"].astype(str))

input_file = "2_wos.json"
output_file = "41_wos_uscfg.json"

with open(input_file, "r", encoding="utf-8") as fin, open(output_file, "w", encoding="utf-8") as fout:
    i=0
    for line in fin:
        obj = json.loads(line)
        pmid = str(obj.get("PMID"))
        if pmid in pmid_set:
            i=i+1
            if i % 10000 == 0:
                print(f"Processed {i} lines")
            fout.write(line)

In [ ]:
from collections import defaultdict
id_to_lines = defaultdict(list)


file_path = "41_wos_uscfg.json"
output_path = "41_wos_uscfg_duplicates.json"

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        obj_id = obj["PMID"]
        id_to_lines[obj_id].append(obj)


with open(output_path, "w", encoding="utf-8") as out:
    for obj_id, records in id_to_lines.items():
        if len(records) > 1:
            for record in records:
                out.write(json.dumps(record) + "\n")

print(f"Total duplicate IDs: {sum(1 for v in id_to_lines.values() if len(v) > 1)}")

In [ ]:

with open("41_wos_uscfg.json", "r") as f:
    data = [json.loads(line) for line in f]


with open("41_wos_uscfg_duplicates.json", "r") as f:
    dup_pmids = {json.loads(line)['PMID'] for line in f}


pmid_to_index = {}


for idx, entry in enumerate(data):
    pmid = entry.get("PMID")
    if pmid in dup_pmids:
        pmid_to_index[pmid] = idx


unique_data = []
for idx, entry in enumerate(data):
    pmid = entry.get("PMID")
    if pmid not in dup_pmids:
        unique_data.append(entry)
    elif pmid_to_index[pmid] == idx:
        unique_data.append(entry)


with open("41_wos_uscfg_deduplicated.json", "w") as f:
    for entry in unique_data:
        f.write(json.dumps(entry) + "\n")

## Add reference counts and identifiers

Attach WoS identifiers, DOI information, and reference counts used later in the impact analyses. These variables require restricted WoS inputs.


In [ ]:
from collections import defaultdict


pmid_to_ids = defaultdict(set)

for file in wos_list:
    i = 0
    year = file[3:7]
    if int(year)<i:
        continue

    start = time.time()
    with gzip.open(os.path.join(wos_path, file), mode="rt", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            pid = obj.get("id")
            if not pid:
                continue
            identifiers = obj.get("identifiers") or []
            for it in identifiers:
                if isinstance(it, dict) and str(it.get("type", "")).strip().lower() == "pmid":
                    pmid = str(it.get("value", "")).strip()
                    if pmid:
                        pmid_to_ids[pmid].add(pid)
    end = time.time()
    print(file, end-start)

In [ ]:
import os, json, gzip
from collections import defaultdict


id_to_pmids = defaultdict(set)
for pmid, pid_list in zip(df["PMID"].astype(str), df["paper_ids"]):
    if isinstance(pid_list, (list, set, tuple)):
        for pid in pid_list:
            if pid is not None:
                id_to_pmids[str(pid)].add(pmid)


pmid_ref_counts = defaultdict(int)
pmid_to_dois = defaultdict(set)
pmid_to_xref_dois = defaultdict(set)

for file in wos_list:
    i = 0
    year = file[3:7]
    if int(year)<i:
        continue

    start = time.time()
    path = os.path.join(wos_path, file)
    with gzip.open(path, mode="rt", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
            except Exception:
                continue


            refs = obj.get("references") or []
            if isinstance(refs, list) and refs:
                pmids_hit_this_line = set()
                for ref in refs:
                    if not isinstance(ref, dict):
                        continue
                    rid = ref.get("id")
                    if not rid:
                        continue
                    rid = str(rid)
                    pmids = id_to_pmids.get(rid)
                    if pmids:
                        pmids_hit_this_line.update(pmids)
                for pmid in pmids_hit_this_line:
                    pmid_ref_counts[pmid] += 1


            cur_id = obj.get("id")
            if cur_id:
                cur_id = str(cur_id)
                pmids_for_cur = id_to_pmids.get(cur_id)
                if pmids_for_cur:
                    identifiers = obj.get("identifiers") or []
                    if isinstance(identifiers, list):
                        for it in identifiers:
                            if not isinstance(it, dict):
                                continue
                            t = str(it.get("type", "")).strip().lower()
                            v = it.get("value")
                            if v is None:
                                continue
                            v = str(v)
                            if not v:
                                continue
                            if t == "doi":
                                for pmid in pmids_for_cur:
                                    pmid_to_dois[pmid].add(v)
                            elif t == "xref_doi":
                                for pmid in pmids_for_cur:
                                    pmid_to_xref_dois[pmid].add(v)

    end = time.time()
    print(file, end-start)


In [ ]:

df["pmid_ref_line_count"] = (
    df["PMID"].astype(str).map(lambda x: pmid_ref_counts.get(x, 0)).astype(int)
)


df["doi"] = df["PMID"].astype(str).map(lambda x: sorted(pmid_to_dois.get(x, set())))
df["xref_doi"] = df["PMID"].astype(str).map(lambda x: sorted(pmid_to_xref_dois.get(x, set())))

In [ ]:
def has_value(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return False
    if isinstance(x, (list, set, tuple)):
        return len(x) > 0
    if isinstance(x, str):
        return x.strip() != ""
    return False

has_doi = df["doi"].apply(has_value)
has_xref = df["xref_doi"].apply(has_value)
has_either = has_doi | has_xref

n_has_doi = int(has_doi.sum())
n_has_xref = int(has_xref.sum())
n_has_either = int(has_either.sum())

print("PMID with ≥1 DOI:", n_has_doi)
print("PMID with ≥1 xref_doi:", n_has_xref)
print("PMID with ≥1 DOI or xref_doi:", n_has_either)

In [ ]:
input_file = "41_pubmed_uscfg.json"


pmid_set = set()
with open(input_file, "r", encoding="utf-8") as fin:
    for line in fin:
        obj = json.loads(line)
        pmid_set.add(obj.get("PMID"))


df_pmid_set = set(df["PMID"].astype(str))


if pmid_set == df_pmid_set:
    print("The two PMID sets match exactly")
else:

    missing_in_df = pmid_set - df_pmid_set

    missing_in_file = df_pmid_set - pmid_set
    
    print("The two PMID sets do not match")
    print("Present in file but missing from df:", missing_in_df)
    print("Present in df but missing from file:", missing_in_file)

## Assign funding and disease labels

Map classified funders to publications, distinguish public, non-public, and mixed funding, attach policy citation indicators, and assign first-level MeSH disease categories. The resulting publication-level analytical file is restricted and is not included.


In [ ]:
from tqdm import tqdm


pmid_records = []
agency_counter = {}


json_path = "41_wos_uscfg_deduplicated.json"
with open(json_path, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Processing JSON"):
        data = json.loads(line)
        pmid = data.get("PMID")
        grants = data.get("grants", [])

        matched_ids = []
        for g in grants:
            agency = g.get("agency")
            if agency in agency_to_id:
                matched_id = agency_to_id[agency]
                matched_ids.append(matched_id)
                agency_counter[matched_id] = agency_counter.get(matched_id, 0) + 1


        pmid_records.append({
            "PMID": pmid,
            "matched_IDs": matched_ids,
            "matched_count": len(matched_ids)
        })


pmid_agencies_df = pd.DataFrame(pmid_records)


agency_freq_df = pd.DataFrame([
    {"ID": k, "frequency": v} for k, v in agency_counter.items()
])
agency_freq_df = agency_freq_df.merge(df_5, on="ID", how="left")


#pmid_agencies_df.to_csv("pmid_agencies_df.csv", index=False)
#agency_freq_df.to_csv("agency_freq_df.csv", index=False)

print("Created pmid_agencies_df and agency_freq_df")

In [ ]:
df_0['PMID'] = df_0['PMID'].astype('int64')
pmid_agencies_df['PMID'] = pmid_agencies_df['PMID'].astype('int64')


merged_df = pd.merge(
    df_0,
    pmid_agencies_df,
    on='PMID',
    how='inner',
    suffixes=('_df0', '_pmid')
)

print(f"Merge complete: {len(merged_df)} rows")
print(f"Merged columns: {len(merged_df.columns)}")

In [ ]:
temp_counts = merged_df['matched_count']


grouped = temp_counts.apply(lambda x: str(x) if x <= 10 else '10+')


count_stats = (
    grouped.value_counts(dropna=False)
    .rename_axis('matched_count')
    .reset_index(name='frequency')
)


count_stats['sort_key'] = count_stats['matched_count'].replace('10+', np.inf).astype(float)
count_stats = count_stats.sort_values('sort_key').drop(columns='sort_key')


total = count_stats['frequency'].sum()
count_stats['percentage'] = (count_stats['frequency'] / total * 100).round(2).astype(str) + '%'


total_row = pd.DataFrame({
    'matched_count': ['Total'],
    'frequency': [total],
    'percentage': ['100.0%']
})
count_stats = pd.concat([count_stats, total_row], ignore_index=True)
count_stats

In [ ]:

def try_eval_list(x):
    if isinstance(x, str):
        s = x.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                return eval(s)
            except Exception:
                return []
        elif s == "" or s.lower() in ("nan", "none"):
            return []
        else:
            return [s]
    return x


for col in merged_df.columns:

    sample = merged_df[col].astype(str).head(200).tolist()
    if any(s.strip().startswith("[") and s.strip().endswith("]") for s in sample):
        print(f"Restoring list values in column: {col}")
        merged_df[col] = merged_df[col].apply(try_eval_list)

In [ ]:
from collections import Counter

merged_df_1 = merged_df[['PMID','file','pmid_ref_line_count','doi','xref_doi','matched_IDs','matched_count']]

cat_dict = df_5.set_index('ID')['category'].to_dict()
cty_dict = df_5.set_index('ID')['country'].to_dict()


def count_categories(ids):
    if not isinstance(ids, list) or len(ids) == 0:
        return pd.Series([0,0,0,0,0,0])
    cats = [cat_dict.get(i) for i in ids if i in cat_dict]
    ctys = [cty_dict.get(i) for i in ids if i in cty_dict]
    c = Counter(cats)
    d = Counter(ctys)
    return pd.Series([
        c.get('Public Agency', 0),
        c.get('Private Company', 0),
        c.get('University', 0),
        c.get('Other Organization', 0),
        d.get('US', 0),
        d.get('Non-US', 0),
        d.get('Uncertain', 0)
    ])


merged_df_1[['govt','company','univ','other','us','non_us','unc_us']] = merged_df_1['matched_IDs'].apply(count_categories)


merged_df_1[['govt','company','univ','other','us','non_us','unc_us']] = merged_df_1[['govt','company','univ','other','us','non_us','unc_us']].fillna(0).astype(int)

In [ ]:
cat_cols = ['govt', 'company', 'univ', 'other']


only_govt = (merged_df_11['govt'] > 0) & (merged_df_11[['company','univ','other']].sum(axis=1) == 0)
govt_and_other = (merged_df_11['govt'] > 0) & (merged_df_11[['company','univ','other']].sum(axis=1) > 0)
no_govt = (merged_df_11['govt'] == 0)


merged_df_11['inst_type'] = np.select(
    [only_govt, govt_and_other, no_govt],
    ['public', 'public+private', 'private'],
    default='uncategorized'
)


merged_df_11['inst_type'].value_counts()

In [ ]:
from collections import defaultdict
from tqdm import tqdm
tqdm.pandas()

cols = ['2017','2018','2019','2020','2021','2022','total']
doi_list = df_o['doi'].fillna("").astype(str).values
doi_values = df_o[cols].values


prefix_len = 8
prefix_dict = defaultdict(list)
for i, d in enumerate(doi_list):
    prefix_dict[d[:prefix_len]].append(i)

def sum_doi_counts_prefix(row):
    dois = []
    for field in ['doi', 'xref_doi']:
        v = row.get(field)
        if isinstance(v, list):
            dois.extend(v)
        elif isinstance(v, str) and len(v) > 0:
            dois.append(v)

    sums = np.zeros(len(cols), dtype=int)
    for d in dois:
        d = str(d).strip()
        if not d or d.lower() in {"", "nan", "none", "0", "null"}:
            continue
        for idx in prefix_dict.get(d[:prefix_len], []):
            if doi_list[idx].startswith(d):
                sums += doi_values[idx]
    return pd.Series(sums, index=[f'{c}_sum' for c in cols])

result = merged_df_11.progress_apply(sum_doi_counts_prefix, axis=1)
merged_df_2 = pd.concat([merged_df_11, result], axis=1)

In [ ]:
merged_df_21 = merged_df_2[['PMID', 'pmid_ref_line_count', 'matched_IDs',
       'matched_count', 'govt', 'company', 'univ', 'other', 'us', 'non_us',
       'unc_us', 'inst_type', '2017_sum', '2018_sum',
       '2019_sum', '2020_sum', '2021_sum', '2022_sum', 'total_sum']]

In [ ]:
cols = ['2017','2018','2019','2020','2021','2022','total']
doi_dict = df_o.set_index('doi')[cols].to_dict(orient='index')


def sum_doi_counts(row):

    dois = []
    for field in ['doi', 'xref_doi']:
        v = row.get(field)
        if isinstance(v, list):
            dois.extend(v)
        elif isinstance(v, str) and v.startswith('['):
            try:
                dois.extend(eval(v))
            except Exception:
                pass
        elif isinstance(v, str) and len(v) > 0:
            dois.append(v)


    sums = np.zeros(len(cols), dtype=int)


    for d in dois:
        if d in doi_dict:
            vals = doi_dict[d]
            sums += np.array([vals[c] for c in cols], dtype=int)
    return pd.Series(sums, index=[f'{c}_sum' for c in cols])


merged_df_2 = pd.concat([merged_df_11, merged_df_11.apply(sum_doi_counts, axis=1)], axis=1)

In [ ]:
def is_nonempty_list(x):
    return isinstance(x, list) and len(x) > 0

doi_nonempty = merged_df[merged_df['matched_count']>0]['doi'].apply(is_nonempty_list)
xref_nonempty = merged_df[merged_df['matched_count']>0]['xref_doi'].apply(is_nonempty_list)


both_empty = ((~doi_nonempty) & (~xref_nonempty)).sum()
doi_only = (doi_nonempty & (~xref_nonempty)).sum()
xref_only = ((~doi_nonempty) & xref_nonempty).sum()
both_have = (doi_nonempty & xref_nonempty).sum()


stats_df = pd.DataFrame({
    'Condition': ['Both empty', 'Only DOI', 'Only XREF_DOI', 'Both have'],
    'Count': [both_empty, doi_only, xref_only, both_have]
})

total = len(merged_df[merged_df['matched_count']>0])
stats_df['Percentage'] = (stats_df['Count'] / total * 100).round(2).astype(str) + '%'


total_row = pd.DataFrame({
    'Condition': ['Total'],
    'Count': [total],
    'Percentage': ['100.0%']
})
stats_df = pd.concat([stats_df, total_row], ignore_index=True)

print("DOI / XREF_DOI missing-value summary:")
stats_df

In [ ]:
merge_x["year"] = merge_x["file"].str.extract(r"WR_(\d{4})_")


file_idx = merge_x.columns.get_loc("file")
cols = merge_x.columns[:file_idx+1].tolist() + ["year"] + merge_x.columns[file_idx+1:-1].tolist()
merge_x = merge_x[cols]

In [ ]:
def get_inst_type_v2(row):
    parts = []

    if row["govt"] != 0:
        parts.append("public")

    if row["company"] != 0:
        parts.append("company")

    if row["univ"] != 0 or row["other"] != 0:
        parts.append("other")
    return "+".join(parts)


merge_x1["inst_type_v2"] = merge_x1.apply(get_inst_type_v2, axis=1)

merge_x1[["govt", "company", "univ", "other", "inst_type_v2"]].head()

In [ ]:
def get_country_type(row):
    parts = []

    if row["us"] != 0 or row["unc_us"] != 0:
        parts.append("us")

    if row["non_us"] != 0:
        parts.append("other")
    return "+".join(parts)


merge_x1["country_type"] = merge_x1.apply(get_country_type, axis=1)

In [ ]:
from ast import literal_eval



df_main = merge_x1


if df_main["matched_IDs"].dtype == object and isinstance(df_main["matched_IDs"].iloc[0], str):
    df_main = df_main.copy()
    df_main["matched_IDs"] = df_main["matched_IDs"].apply(
        lambda x: literal_eval(x) if isinstance(x, str) else x
    )


dict_df = df_5[["ID", "category", "country"]].copy()


dict_df["is_us"] = dict_df["country"].isin(["US", "Uncertain"])
dict_df["is_other"] = dict_df["country"].eq("Non-US")
dict_df["is_govt"] = dict_df["category"].str.contains(r"\bPublic Agency\b", case=False, na=False)


mid = (
    df_main[["PMID", "matched_IDs"]]
    .explode("matched_IDs")
    .dropna(subset=["matched_IDs"])
    .rename(columns={"matched_IDs": "ID"})
)


mid["ID"] = pd.to_numeric(mid["ID"], errors="coerce").dropna().astype(int)

merged = mid.merge(dict_df[["ID", "is_us", "is_other", "is_govt"]],
                   on="ID", how="left")


merged["us_govt"] = merged["is_us"] & merged["is_govt"]
merged["other_govt"] = merged["is_other"] & merged["is_govt"]


agg = merged.groupby("PMID").agg(
    us_any=("is_us", "any"),
    other_any=("is_other", "any"),
    us_govt_any=("us_govt", "any"),
    other_govt_any=("other_govt", "any"),
)


def build_country_type(row):
    parts = []
    if row.us_any:
        parts.append("us")
    if row.other_any:
        parts.append("other")
    return "+".join(parts)

def build_country_type_v2(row):
    parts = []
    if row.us_any:
        parts.append("us(govt)" if row.us_govt_any else "us")
    if row.other_any:
        parts.append("other(govt)" if row.other_govt_any else "other")
    return "+".join(parts)

agg["country_type_recalc"] = agg.apply(build_country_type, axis=1)
agg["country_type_v2"] = agg.apply(build_country_type_v2, axis=1)


df_main = df_main.merge(
    agg[["country_type_recalc", "country_type_v2"]],
    left_on="PMID", right_index=True, how="left"
)

In [ ]:
df_main_2 = df_main[['PMID', 'year', 'pmid_ref_line_count', 'matched_IDs', 'matched_count',
       'govt', 'company', 'univ', 'other', 'us', 'non_us', 'unc_us',
       'inst_type','inst_type_v2', 'country_type', 'country_type_v2','2017_sum', '2018_sum', '2019_sum', '2020_sum', '2021_sum',
       '2022_sum', 'total_sum']]

In [ ]:
ui_to_mn = {}


for i in range(len(mesh_data)):
    ui = mesh_data.loc[i, 'UI']
    mn = mesh_data.loc[i, 'MN']


    if pd.isna(mn):
        continue


    mn_list = [m.strip() for m in mn.split(';')]
    filtered = [m for m in mn_list if m.startswith('C') or m.startswith('F03')]


    if filtered:
        ui_to_mn[ui] = filtered

In [ ]:
input_file = "41_pubmed_uscfg.json"
output_file = "a1_mesh_disease.json"

pmid_to_mn = {}
i=0
with open(input_file, "r", encoding="utf-8") as fin:
    for line in fin:
        obj = json.loads(line)
        i=i+1
        if i%10000==0:
            print(i)
        pmid = obj.get("PMID")
        if not pmid:
            continue

        mn_set = set()
        if "MeshHeadingList" in obj:
            for x in obj["MeshHeadingList"]:
                if "DescriptorName" in x and "@UI" in x["DescriptorName"]:
                    ui = x["DescriptorName"]["@UI"]
                    if ui in ui_to_mn:
                        mn_set.update(ui_to_mn[ui])

        if mn_set:
            pmid_to_mn[pmid] = list(mn_set)


with open(output_file, "w", encoding="utf-8") as fout:
    json.dump(pmid_to_mn, fout, ensure_ascii=False, indent=2)

In [ ]:
pmid_to_lvl1 = {}
pmid_to_lvl2 = {}

for pmid, mn_list in pmid_to_mn.items():
    lvl1_list = []
    lvl2_list = []

    for m in mn_list:
        parts = m.split(".")
        if len(parts) >= 1:
            lvl1_list.append(parts[0])
        if len(parts) >= 2:
            lvl2_list.append(".".join(parts[:2]))
        elif len(parts) == 1:
            lvl2_list.append(parts[0])

    pmid_to_lvl1[pmid] = sorted(list(set(lvl1_list)))
    pmid_to_lvl2[pmid] = sorted(list(set(lvl2_list)))


with open("a1_mesh_disease_lvl1.json", "w", encoding="utf-8") as f1:
    json.dump(pmid_to_lvl1, f1, ensure_ascii=False, indent=2)

with open("a1_mesh_disease_lvl2.json", "w", encoding="utf-8") as f2:
    json.dump(pmid_to_lvl2, f2, ensure_ascii=False, indent=2)

In [ ]:
df_lvl1 = pd.DataFrame([
    {'PMID': int(k), 'mesh_lvl1': v}
    for k, v in pmid_to_lvl1.items()
])
df_merged = df_main_2.merge(df_lvl1, on='PMID', how='left')
df_lvl2 = pd.DataFrame([
    {'PMID': int(k), 'mesh_lvl2': v}
    for k, v in pmid_to_lvl2.items()
])
df_merged = df_merged.merge(df_lvl2, on='PMID', how='left')
df_merged